# Step 1.2 - Keep only phenotypically active compounds

**What this notebook does.** It removes compound wells whose Cell Painting profile cannot be told apart
from untreated (DMSO) cells, for the Bayer DINO and OpenPhenom embeddings.

**Why.** Many compounds have no visible effect on cells at the tested dose. Their profiles are
essentially measurement noise. Keeping them dilutes comparisons between representations: two inactive
compounds may look "similar" only because both look like untreated cells.

**How - the Phenotypic Activity Score.**
1. Take the median DMSO profile as the "untreated" reference point.
2. Measure every well's Euclidean distance from that point. This is meaningful because the embeddings
   are already sphered (Step 1.1), so all directions count equally.
3. Call a compound well **active** if it lies further from the reference than 95% of DMSO wells do
   (threshold = 95th percentile of the DMSO wells' own distances).

**Inputs.** `BAYER_DINO_embeddings_normed_final.parquet` (Step 0.3),
`openphenom_{384,1920}_sphered_mad_robustized.parquet` (Step 1.1).

**Outputs.** Active-only versions of each file (names given in each part).

Filtering is done per **well**, not per compound: a compound can keep some wells (e.g. a high dose)
and lose others.

**Where to run it.** Written for Google Colab / Colab Enterprise: it uses the `gsutil` command-line tool and
stages temporary files under `/content/` (Colab's working disk). You need read/write access to the
`gs://calico-reference-embeddings` Cloud Storage bucket.

## Setup

All libraries used in this notebook are imported in the next cell. Locally they are installed by `uv sync`
(see the main README); on Google Colab, first run
`!pip install git+https://github.com/axiomcura/ModelBenchmarking.git`, which installs the same dependencies.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.spatial.distance import cdist

## Part A - Bayer DINO (normalized)

> ⚠️ **Notes on this part**
> * **Output file name.** A.6 writes
>   `Bayer_DINO_embeddings__Kim_et_al_subset__sphering_mad_robustize_well_features__ACTIVE_ONLY-test2.parquet`,
>   whereas Steps 3 and 4 read `Bayer_DINO_embeddings_normed_final-actives_only.parquet`. The file used
>   downstream was produced by another run or version of this code.
> * **Which columns count as features.** Distances are computed over *all numeric columns*
>   (`select_dtypes(include=number)`). If the table contains numeric metadata (for example a
>   `concentration` column), it is included in the distance.
> * The saved table keeps the `Phenotypic_Activity_Score` column (Part B removes it), so later code that
>   treats every numeric column as a feature would include the score.

### A.1 Configuration

In [2]:
PARQUET_FILE_PATH = 'gs://calico-reference-embeddings/Annotations_from_Roman/Bayer_data_harmonized/Bayer_DINO_embeddings/BAYER_DINO_embeddings_normed_final.parquet'

# Output path for the filtered dataset
OUTPUT_PARQUET_PATH = 'gs://calico-reference-embeddings/Annotations_from_Roman/Bayer_data_harmonized/Bayer_DINO_embeddings/Bayer_DINO_embeddings__Kim_et_al_subset__sphering_mad_robustize_well_features__ACTIVE_ONLY-test2.parquet'

CONTROL_COLUMN = 'control_type'
DMSO_LABEL = 'negcon'

### A.2 Load the normalized embeddings

In [3]:
print(f"📂 Loading pre-sphered embeddings from GCS...")
df_bayer = pd.read_parquet(PARQUET_FILE_PATH)
print(f"   ✅ Loaded {len(df_bayer):,} rows and {df_bayer.shape[1]:,} columns.")

### A.3 Separate DMSO controls from compound wells

In [4]:
print("\n⚙️ Separating controls from perturbations...")

mask_dmso = df_bayer[CONTROL_COLUMN] == DMSO_LABEL
df_dmso = df_bayer[mask_dmso].copy()
df_drugs = df_bayer[~mask_dmso].copy()

# Isolate just the numerical embedding features
dmso_embs = df_dmso.select_dtypes(include=[np.number]).values
drug_embs = df_drugs.select_dtypes(include=[np.number]).values

print(f"   -> Found {len(df_dmso):,} DMSO controls.")
print(f"   -> Found {len(df_drugs):,} Drug perturbations.")

### A.4 Phenotypic activity score
Euclidean distance of every well from the median DMSO profile.

In [5]:
print("\n📏 Calculating distances for already-sphered data...")

# 1. Find the exact center of the DMSO sphere
dmso_centroid = np.median(dmso_embs, axis=0).reshape(1, -1)

# 2. Calculate pure Euclidean distance to the center
dmso_pas = cdist(dmso_embs, dmso_centroid, metric='euclidean').flatten()
drug_pas = cdist(drug_embs, dmso_centroid, metric='euclidean').flatten()

df_dmso['Phenotypic_Activity_Score'] = dmso_pas
df_drugs['Phenotypic_Activity_Score'] = drug_pas

### A.5 Keep wells beyond the 95th percentile of DMSO

In [6]:
THRESHOLD_PERCENTILE = 95
activity_threshold = np.percentile(dmso_pas, THRESHOLD_PERCENTILE)

print(f"\n🛑 Setting Activity Threshold at the {THRESHOLD_PERCENTILE}th percentile of DMSO.")
print(f"   -> Threshold Distance: {activity_threshold:.2f}")

# Filter the drugs
active_mask = df_drugs['Phenotypic_Activity_Score'] > activity_threshold
df_active_drugs = df_drugs[active_mask].copy()
inactive_count = len(df_drugs) - len(df_active_drugs)

print("\n🏆 FILTERING RESULTS")
print("-" * 50)
print(f"   Total Drugs:        {len(df_drugs):,}")
print(f"   ✅ Active (Kept):   {len(df_active_drugs):,} ({(len(df_active_drugs)/len(df_drugs))*100:.1f}%)")
print(f"   ❌ Inactive (Drop): {inactive_count:,} ({(inactive_count/len(df_drugs))*100:.1f}%)")
print("-" * 50)

### A.6 Save the active wells

In [7]:
print(f"\n💾 Saving active-only filtered embeddings to Google Cloud Storage...")
# Snappy compression is excellent for Parquet files
df_active_drugs.to_parquet(OUTPUT_PARQUET_PATH, compression='snappy')
print(f"🎉 Success! Filtered dataset permanently saved to:")
print(f"   {OUTPUT_PARQUET_PATH}")

### A.7 Plot the score distributions
DMSO wells (grey) vs compound wells (teal), with the threshold in red.

In [8]:
plt.figure(figsize=(10, 6))

sns.kdeplot(dmso_pas, fill=True, color='gray', label='DMSO Controls (Noise)')
sns.kdeplot(drug_pas, fill=True, color='teal', label='Drug Perturbations')

plt.axvline(activity_threshold, color='red', linestyle='--',
            label=f'{THRESHOLD_PERCENTILE}th Percentile Threshold')

plt.title('Phenotypic Activity Score (Sphered Euclidean Distance)', fontsize=16)
plt.xlabel('Distance from DMSO Centroid', fontsize=12)
plt.ylabel('Density', fontsize=12)
plt.legend()
sns.despine()
plt.tight_layout()
plt.show()

## Part B - OpenPhenom (normalized)

**Run this part twice.** It is currently set to the 1,920-dimensional embeddings; the 384-dimensional input and
output paths are the commented lines in B.1.

### B.1 Configuration

In [9]:
# Input: The Sphered and Robustized OpenPhenom Parquet
#INPUT_PARQUET = 'gs://calico-reference-embeddings/embeddings/JUMP_Bayer_compounds/OpenPhenom/parquets/openphenom_384_sphered_mad_robustized.parquet'
INPUT_PARQUET = 'gs://calico-reference-embeddings/embeddings/JUMP_Bayer_compounds/OpenPhenom/parquets/openphenom_1920_sphered_mad_robustized.parquet'

# Output: The Active-Only subset
#OUTPUT_PARQUET = 'gs://calico-reference-embeddings/embeddings/JUMP_Bayer_compounds/OpenPhenom/parquets/openphenom_384_sphered_mad_robustized-ACTIVE.parquet'
OUTPUT_PARQUET =  'gs://calico-reference-embeddings/embeddings/JUMP_Bayer_compounds/OpenPhenom/parquets/openphenom_1920_sphered_mad_robustized-ACTIVE.parquet'

# Universal DMSO Identifiers
DMSO_INCHIKEY = 'IAZDPXIOMUYVGZ-UHFFFAOYSA-N'
DMSO_JCP_ID = 'JCP2022_033924'

### B.2 Load the normalized embeddings

In [10]:
print("📂 Loading Sphered OpenPhenom embeddings from GCS...")
df = pd.read_parquet(INPUT_PARQUET)

print(f"   ✅ Loaded {len(df):,} rows and {df.shape[1]:,} columns.")

# Isolate metadata vs numerical embeddings
meta_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()
emb_cols = df.select_dtypes(include=[np.number]).columns.tolist()

### B.3 Find the DMSO control wells
By the DMSO InChIKey (the index) and the JUMP DMSO identifier.

In [11]:
print("\n🔍 Locating DMSO negative controls using chemical identifiers...")

# We check both the InChIKey (Index) and any JCP ID columns to ensure we catch all DMSO rows
is_dmso = pd.Series(False, index=df.index)

# 1. Check the Index (if it's InChIKey)
if df.index.name == 'InChIKey':
    is_dmso = is_dmso | (df.index == DMSO_INCHIKEY)

# 2. Check metadata columns for the JCP ID
for col in meta_cols:
    is_dmso = is_dmso | (df[col].astype(str).str.strip().str.upper() == DMSO_JCP_ID.upper())

df_dmso = df[is_dmso].copy()
df_drugs = df[~is_dmso].copy()

dmso_count = len(df_dmso)
print(f"   -> Found {dmso_count:,} DMSO control well(s).")
print(f"   -> Found {len(df_drugs):,} Drug perturbations.")

if dmso_count == 0:
    raise ValueError(f"❌ No DMSO controls found! Could not find '{DMSO_INCHIKEY}' or '{DMSO_JCP_ID}'.")

# Extract the purely numerical float arrays
dmso_embs = df_dmso[emb_cols].values
drug_embs = df_drugs[emb_cols].values

### B.4 Phenotypic activity score

In [12]:
print("\n📏 Calculating Phenotypic Activity Scores (Euclidean Distance)...")

# 1. Find the exact mathematical center of the DMSO baseline
dmso_centroid = np.median(dmso_embs, axis=0).reshape(1, -1)

# 2. Calculate the pure Euclidean distance to the DMSO center
# Because the data is already sphered, Euclidean distance is mathematically perfect here!
dmso_pas = cdist(dmso_embs, dmso_centroid, metric='euclidean').flatten()
drug_pas = cdist(drug_embs, dmso_centroid, metric='euclidean').flatten()

df_dmso['Phenotypic_Activity_Score'] = dmso_pas
df_drugs['Phenotypic_Activity_Score'] = drug_pas

### B.5 Choose the threshold and filter
With at least 10 DMSO wells the threshold is the 95th percentile of their distances. With fewer, the 5% of compounds closest to DMSO are removed instead - a trim, not a statistical test of activity.

In [13]:
# If we have a healthy population of DMSO replicates, use the 95th percentile of DMSO noise
if dmso_count >= 10:
    THRESHOLD_PERCENTILE = 95
    activity_threshold = np.percentile(dmso_pas, THRESHOLD_PERCENTILE)
    print(f"\n🛑 Setting Activity Threshold at the {THRESHOLD_PERCENTILE}th percentile of DMSO noise.")

# If the dataset was pre-aggregated and we only have 1 or a few DMSO rows, we drop the bottom 5% of drugs
else:
    DROP_BOTTOM_PERCENT = 5
    activity_threshold = np.percentile(drug_pas, DROP_BOTTOM_PERCENT)
    print(f"\n🛑 Only {dmso_count} DMSO row(s) detected. Setting threshold to drop the bottom {DROP_BOTTOM_PERCENT}% of inactive drugs.")

print(f"   -> Threshold Distance: {activity_threshold:.2f}")

# Filter the drugs to keep ONLY the active ones
active_mask = df_drugs['Phenotypic_Activity_Score'] > activity_threshold

# Apply to the original dataframe to preserve all pristine formatting/columns
df_active_drugs = df_drugs[active_mask].copy()

inactive_count = len(df_drugs) - len(df_active_drugs)

print("\n🏆 FILTERING RESULTS")
print("-" * 50)
print(f"   Total Drugs:        {len(df_drugs):,}")
print(f"   ✅ Active (Kept):   {len(df_active_drugs):,} ({(len(df_active_drugs)/len(df_drugs))*100:.1f}%)")
print(f"   ❌ Inactive (Drop): {inactive_count:,} ({(inactive_count/len(df_drugs))*100:.1f}%)")
print("-" * 50)

### B.6 Save the active wells
The `Phenotypic_Activity_Score` column is removed first so it is never mistaken for an embedding dimension.

In [14]:
print(f"\n💾 Saving active-only filtered embeddings to Google Cloud Storage...")

# Drop the PAS score before saving so it doesn't get accidentally treated as an embedding dimension
df_active_drugs = df_active_drugs.drop(columns=['Phenotypic_Activity_Score'], errors='ignore')

# Save securely to GCS
df_active_drugs.to_parquet(OUTPUT_PARQUET, compression='snappy')
print(f"🎉 Success! Filtered dataset permanently saved to:\n   {OUTPUT_PARQUET}")

### B.7 Plot the score distributions

In [15]:
plt.figure(figsize=(10, 6))

# If there is enough DMSO to form a distribution, plot it
if dmso_count > 1:
    sns.kdeplot(dmso_pas, fill=True, color='gray', label='DMSO Controls (Noise)', cut=0)

sns.kdeplot(drug_pas, fill=True, color='teal', label='Drug Perturbations', cut=0)

plt.axvline(activity_threshold, color='red', linestyle='--', linewidth=2,
            label=f'Activity Threshold ({activity_threshold:.2f})')

plt.title('OpenPhenom Phenotypic Activity Score Distribution', fontsize=16, pad=15)
plt.xlabel('Euclidean Distance from DMSO Baseline', fontsize=12)
plt.ylabel('Density', fontsize=12)
plt.legend()
sns.despine()
plt.tight_layout()
plt.show()